In [1]:
import pandas as pd

# ── 1. Load data ──────────────────────────────────────────────────────────────
df = pd.read_csv('/content/data_sintesis_wellbeing_clean.csv', encoding='utf-8-sig')

print("Kolom sebelum:")
for col in df.columns:
    print(f"  - {col}")

# ── 2. Hapus kolom yang tidak digunakan ───────────────────────────────────────
kolom_hapus = [
    'Usia Anda',
    'Jenis Kelamin',
    'Seberapa Tertekan Anda (1-5)',
    'Seberapa Sering Merasa Lelah Emosional (1-5)',
    'Seberapa Termotivasi Anda (1-5)',
]

df = df.drop(columns=kolom_hapus)

print("\nKolom sesudah:")
for col in df.columns:
    print(f"  - {col}")

# ── 3. Simpan ─────────────────────────────────────────────────────────────────
df.to_csv('data_sintesis_wellbeing_final.csv', index=False, encoding='utf-8-sig')

print(f"\nShape akhir : {df.shape[0]} baris × {df.shape[1]} kolom")
print("File tersimpan: data_sintesis_wellbeing_final.csv")

Kolom sebelum:
  - Usia Anda
  - Jenis Kelamin
  - Jam Tidur Semalam
  - Seberapa Sibuk Anda Hari Ini (1-5)
  - Seberapa Banyak Tugas/Pekerjaan (1-5)
  - Seberapa Tertekan Anda (1-5)
  - Suasana Hati Anda (1-5)
  - Seberapa Sering Merasa Lelah Emosional (1-5)
  - Seberapa Termotivasi Anda (1-5)
  - Ceritakan Hari Anda Hari Ini

Kolom sesudah:
  - Jam Tidur Semalam
  - Seberapa Sibuk Anda Hari Ini (1-5)
  - Seberapa Banyak Tugas/Pekerjaan (1-5)
  - Suasana Hati Anda (1-5)
  - Ceritakan Hari Anda Hari Ini

Shape akhir : 3000 baris × 5 kolom
File tersimpan: data_sintesis_wellbeing_final.csv


In [4]:
import pandas as pd
import numpy as np

# ── 1. Load data ──────────────────────────────────────────────────────────────
df = pd.read_csv('/content/dataset_model2_with_emotions.csv')

# ── 2. Hapus kolom yang tidak digunakan ───────────────────────────────────────
kolom_hapus = [
    'Seberapa Tertekan Anda (1-5)',
    'Seberapa Sering Merasa Lelah Emosional (1-5)',
    'Seberapa Termotivasi Anda (1-5)',
    'Wellbeing_Score',    # label lama
    'Wellbeing_Label',    # label lama
]
df = df.drop(columns=kolom_hapus)

# ── 3. Emotion-Based Weighted Scoring ─────────────────────────────────────────
# Emosi POSITIF → skor tinggi = wellbeing baik
positive_score = (
    df['prob_joy'] +
    df['prob_trust'] +
    df['prob_anticipation']
)

# Emosi NEGATIF → skor tinggi = wellbeing buruk
negative_score = (
    df['prob_anger'] +
    df['prob_disgust'] +
    df['prob_fear'] +
    df['prob_sadness']
)

# Selisih positif - negatif → rentang bisa -1 sampai +1
raw_score = positive_score - negative_score

# Normalisasi ke 0-1 agar threshold lebih mudah diinterpretasi
df['Emotion_Score'] = ((raw_score - raw_score.min()) /
                       (raw_score.max() - raw_score.min())).round(4)

# ── 4. Threshold dari Quantile ────────────────────────────────────────────────
# Threshold bukan ditentukan manual/arbitrary, tapi dari distribusi data sendiri
# Persentil 33 = batas Low|Medium, Persentil 66 = batas Medium|High
q33 = df['Emotion_Score'].quantile(0.33)
q66 = df['Emotion_Score'].quantile(0.66)

print("=" * 55)
print("THRESHOLD DARI QUANTILE DATA")
print("=" * 55)
print(f"  Persentil 33 (batas Low | Medium) : {q33:.4f}")
print(f"  Persentil 66 (batas Medium | High): {q66:.4f}")
print(f"\n  → High    : Emotion_Score < {q33:.4f}")
print(f"  → Medium : {q33:.4f} ≤ Emotion_Score ≤ {q66:.4f}")
print(f"  → Low   : Emotion_Score > {q66:.4f}")

# ── 5. Assign Label ───────────────────────────────────────────────────────────
def assign_label(score):
    if score < q33:
        return 'High'
    elif score <= q66:
        return 'Medium'
    else:
        return 'Low'

df['Wellbeing_Label'] = df['Emotion_Score'].apply(assign_label)

# ── 6. Simpan ─────────────────────────────────────────────────────────────────
out = '/content/dataset_final_labeled.csv'
df.to_csv(out, index=False, encoding='utf-8-sig')

# ── 7. Ringkasan ──────────────────────────────────────────────────────────────
print("\n" + "=" * 55)
print("DISTRIBUSI LABEL")
print("=" * 55)
dist = df['Wellbeing_Label'].value_counts().reindex(['Low', 'Medium', 'High'])
for label, count in dist.items():
    pct = count / len(df) * 100
    bar = '█' * int(pct / 2)
    print(f"  {label:<8} {count:>5} ({pct:5.1f}%)  {bar}")

print(f"\nStatistik Emotion_Score per label:")
print(df.groupby('Wellbeing_Label')['Emotion_Score']
        .agg(['min', 'mean', 'max'])
        .rename(columns={'min': 'Min', 'mean': 'Rata-rata', 'max': 'Max'})
        .reindex(['Low', 'Medium', 'High'])
        .round(4).to_string())

print(f"\nKolom akhir:")
for col in df.columns:
    print(f"  - {col}")

print(f"\nShape akhir : {df.shape[0]} baris × {df.shape[1]} kolom")
print(f"File tersimpan: {out}")

THRESHOLD DARI QUANTILE DATA
  Persentil 33 (batas Low | Medium) : 0.0455
  Persentil 66 (batas Medium | High): 0.1615

  → High    : Emotion_Score < 0.0455
  → Medium : 0.0455 ≤ Emotion_Score ≤ 0.1615
  → Low   : Emotion_Score > 0.1615

DISTRIBUSI LABEL
  Low       1015 ( 33.8%)  ████████████████
  Medium    1005 ( 33.5%)  ████████████████
  High       980 ( 32.7%)  ████████████████

Statistik Emotion_Score per label:
                    Min  Rata-rata     Max
Wellbeing_Label                           
Low              0.1896     0.4905  1.0000
Medium           0.0455     0.0822  0.1615
High             0.0000     0.0287  0.0433

Kolom akhir:
  - Jam Tidur Semalam
  - Seberapa Sibuk Anda Hari Ini (1-5)
  - Seberapa Banyak Tugas/Pekerjaan (1-5)
  - Suasana Hati Anda (1-5)
  - Ceritakan Hari Anda Hari Ini
  - prob_anger
  - prob_anticipation
  - prob_disgust
  - prob_fear
  - prob_joy
  - prob_sadness
  - prob_trust
  - Emotion_Score
  - Wellbeing_Label

Shape akhir : 3000 baris × 14 ko